In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from utils.PDFProcessor import PDFProcessor
import os

c:\Users\youss\OneDrive\Desktop\Academic_Knowledge_&_Research_Assistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
pdf_file_path = Path.cwd().parent / "Datasets" / "pdfs" / "2410.01574v4.pdf"
pdf_processor = PDFProcessor()
pages_content, metadata = pdf_processor.process_pdf(str(pdf_file_path))

print(f"Metadata:")
print(f"Title: {metadata.title}")
print(f"Authors: {metadata.author}")
print(f"Creation Date: {metadata.creation_date}")
print(f"Number of Pages: {metadata.num_pages}")
print(f"Language: {metadata.language}")
print(f"Has Text: {metadata.has_text}")
print(f"Requires OCR: {metadata.required_ocr}")

2026-10-08 07:51:16,344 - utils.PDFProcessor - INFO - Processing PDF: 2410.01574v4.pdf
2026-10-08 07:51:16,494 - utils.PDFProcessor - INFO - PDF has 24 pages. Text: True, OCR needed: False
Extracting text: 100%|██████████| 24/24 [00:01<00:00, 16.60it/s]
2026-10-08 07:51:17,944 - utils.PDFProcessor - INFO - Successfully processed 24/24 pages


Metadata:
Title: Adversarial Robustness of AI-Generated Image Detectors in the Real World
Authors: Sina Mavali; Jonas Ricker; David Pape; Asja Fischer; Lea Schönherr
Creation Date: 
Number of Pages: 24
Language: en
Has Text: True
Requires OCR: False


In [3]:
# sample a random page and print its content and statistics
import random

document = random.choice(pages_content)
print(f"Snipet Content of a random page:\n\"{document.text[:100]}\"...")
print(f"Statistics of the random page:")
print(f"Page Number: {document.page_number}")
print(f"Language: {document.language}")
print(f"Raw Length: {document.raw_length}")
print(f"Cleaned Length: {document.cleaned_length}")
print(f"Number of Sentences: {document.num_sentences}")
print(f"Tokens Estimate: {document.tokens_estimate}")

Snipet Content of a random page:
"6
S. Mavali, J. Ricker, D. Pape, A. Fischer, and L. Schönherr
Conv-B and CLIP versions). These detec"...
Statistics of the random page:
Page Number: 6
Language: en
Raw Length: 2863
Cleaned Length: 2862
Number of Sentences: 27
Tokens Estimate: 567.9100000000001


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=50)
texts = text_splitter.split_text(str(document.text))
print(f"Number of chunks after splitting: {len(texts)}")
print(f"Estimated number of tokens of the first chunk: {len(texts[0].split())}")
print(f"Length of the first chunk: {len(texts[0])}")
print(f"First chunk:\n\"{texts[0]}\"...")

Number of chunks after splitting: 4
Estimated number of tokens of the first chunk: 120
Length of the first chunk: 789
First chunk:
"6
S. Mavali, J. Ricker, D. Pape, A. Fischer, and L. Schönherr
Conv-B and CLIP versions). These detectors were chosen to span a diverse range
of concepts, architectures, and training paradigms.
Corvi. This detector leverages a ResNet-50 [21] and relies on heavy data aug-
mentation during training. The notable improvement is the removal of the initial
downsampling layer, which preserves high-frequency information at the cost of
a higher number of trainable parameters.
UnivFD. Ojha et al. [34] address universal fake image detection by leveraging
a large vision foundation model, specifically CLIP-ViT-L/14 [12], as a feature
extractor. A single linear layer is then trained on top of these pre-extracted fea-
tures, while the CLIP weights remain frozen. This decoupled training approach"...


In [5]:
from transformers import AutoTokenizer
from dotenv import load_dotenv
import os

load_dotenv()

HF_TOKEN = os.getenv("HG_ACCESS_TOKEN")

tokenizer = AutoTokenizer.from_pretrained(
    "google/gemma-3-4b-it",
    token=HF_TOKEN
)

def token_length_function(text: str) -> int:
    return len(tokenizer.encode(text, add_special_tokens=False))

token_based_text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, 
    chunk_overlap=50,
    length_function=token_length_function
)

2026-10-08 07:51:33,486 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/google/gemma-3-4b-it "HTTP/1.1 200 OK"
2026-10-08 07:51:34,020 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/google/gemma-3-4b-it/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-10-08 07:51:34,204 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/google/gemma-3-4b-it/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-10-08 07:51:35,772 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/google/gemma-3-4b-it "HTTP/1.1 200 OK"


In [6]:
texts = token_based_text_splitter.split_text(str(document.text))
print(f"Number of chunks after splitting: {len(texts)}")
print(f"Estimated number of tokens of the first chunk: {len(texts[0].split())}")
print(f"Length of the first chunk: {len(texts[0])}")
print(f"First chunk:\n\"{texts[0]}\"...")

Number of chunks after splitting: 2
Estimated number of tokens of the first chunk: 301
Length of the first chunk: 1980
First chunk:
"6
S. Mavali, J. Ricker, D. Pape, A. Fischer, and L. Schönherr
Conv-B and CLIP versions). These detectors were chosen to span a diverse range
of concepts, architectures, and training paradigms.
Corvi. This detector leverages a ResNet-50 [21] and relies on heavy data aug-
mentation during training. The notable improvement is the removal of the initial
downsampling layer, which preserves high-frequency information at the cost of
a higher number of trainable parameters.
UnivFD. Ojha et al. [34] address universal fake image detection by leveraging
a large vision foundation model, specifically CLIP-ViT-L/14 [12], as a feature
extractor. A single linear layer is then trained on top of these pre-extracted fea-
tures, while the CLIP weights remain frozen. This decoupled training approach
improves the detector’s generalization.
DRCT. Diffusion reconstruction contra